# Ejercicio: asistente de documentos con NVIDIA NIM y Gradio

**Tiempo:** 40–45 minutos  
**Grupos:** 3–4 personas

## Contexto

En los notebooks anteriores construimos chatbots que responden preguntas
generales. En este ejercicio construiremos un asistente que responda preguntas
sobre un documento específico: **“Attention Is All You Need”**
(Vaswani et al., 2017).

Inyectar un documento completo en el contexto permite construir un primer
prototipo de *document question answering*. Esto ayuda a entender la motivación
de **RAG**, aunque todavía no es RAG: aquí no existe recuperación de fragmentos.

## Objetivo

Construir una aplicación Gradio donde el usuario pueda:

1. Cargar o leer el PDF del paper.
2. Hacer preguntas sobre su contenido.
3. Obtener respuestas basadas exclusivamente en el documento.

## Lo que van a aprender

- Extracción de texto de PDF con `pypdf`.
- Uso de contexto externo en un `system` prompt.
- Consumo de NVIDIA NIM mediante una API compatible con OpenAI.
- Streaming e historial con Gradio.
- Limitaciones de este enfoque y motivación de RAG.

## Paso 0: instalación y configuración

### 0.1 Instala las dependencias

- `pypdf`: extracción del texto del PDF.
- `gradio`: interfaz web.
- `openai`: cliente compatible con NVIDIA NIM.
- `python-dotenv`: lectura de variables de entorno.

```bash
pip install -q --upgrade pypdf gradio openai python-dotenv
```

In [20]:
pip install -q --upgrade pypdf gradio openai python-dotenv

### 0.2 Descarga el paper

Descarga el PDF abierto desde:

```text
https://arxiv.org/pdf/1706.03762
```

Guárdalo junto al notebook como `attention_is_all_you_need.pdf`.

### 0.3 Configura tus credenciales

Crea un archivo `.env` con tu API key de NVIDIA:

```text
NVIDIA_API_KEY="tu_key_aqui"
```

Puedes crearla en [NVIDIA Build](https://build.nvidia.com/settings/api-keys).

## Paso 1: extracción de texto del PDF

Lee el PDF y extrae su contenido como texto plano.

**Instrucciones:**

1. Importa `PdfReader` desde `pypdf`.
2. Crea `extract_text_from_pdf(pdf_path)`.
3. Abre el PDF e itera sobre `reader.pages`.
4. Concatena el resultado de `page.extract_text()`; contempla páginas sin texto.
5. Retorna el texto completo.
6. Prueba la función e imprime los primeros 500 caracteres.

> **Pista:** usa `page.extract_text() or ""` para evitar valores `None`.

In [36]:
from IPython.utils import text
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text


## Paso 2: inicialización del cliente de NVIDIA NIM

**Instrucciones:**

1. Importa `os`, `load_dotenv` y `OpenAI`.
2. Carga las variables con `load_dotenv()`.
3. Verifica que `NVIDIA_API_KEY` esté disponible sin imprimirla.
4. Inicializa `OpenAI` con:
   - `base_url="https://integrate.api.nvidia.com/v1"`
   - `api_key=os.getenv("NVIDIA_API_KEY")`
5. Define:

```python
MODELO = "nvidia/nemotron-3-super-120b-a12b"
```

In [37]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY")
)
MODELO = "nvidia/nemotron-3-super-120b-a12b"


## Paso 3: función de chat con el documento

Construye un `system` prompt que incluya el texto del paper y obligue al modelo
a fundamentar sus respuestas en él.

**Instrucciones:**

1. Crea `build_system_prompt(document_text)` para:
   - definir al asistente como experto en el paper;
   - incluir el documento entre delimitadores claros;
   - exigir respuestas basadas únicamente en el documento;
   - indicar que responda “No encuentro esa información en el documento” cuando corresponda.

2. Crea `history_to_messages(history)` para convertir el historial de Gradio a
   mensajes con roles `user` y `assistant`.

3. Crea `chat_con_documento(message, history, document_text)` para:
   - iniciar `messages` con el `system` prompt;
   - agregar el historial y el mensaje actual;
   - llamar `client.chat.completions.create(..., stream=True)`;
   - acumular `chunk.choices[0].delta.content`;
   - usar `yield` para entregar la respuesta progresivamente.

> **Pista:** utiliza `temperature=1.0` y `top_p=0.95`, valores recomendados para
> Nemotron 3 Super.

In [55]:
def build_system_prompt(document_text):
    return f"""Eres un asistente especializado que responde preguntas sobre un
    documento especifico que veras a continuacion entre <documento> y
    </documento>. Debes seguir las siguientes reglas de forma OBLIGATORIA:
    1. Tienes que responder UNICAMENTE en base a la información obtenida de
    este documento
    2. Cuandos ea posible, cita y parafrasea directamente el documento
    3. Si una pregunta no se puede inferir a partir de la informacion del
    documento debes responder especificamente“No encuentro esa información en el
    documento”
    <documento>
    {document_text}
    </documento>"""

def history_to_messages(history):
    messages = []
    for item in history:
        if isinstance(item, list):
          messages.append({"role": item["user"], "content": item["content"]})
        else:
          user_msg, assistant_msg = item
          if user_msg: messages.append({"role": "user", "content": user_msg})
          if assistant_msg: messages.append({"role": "assistant", "content": assistant_msg})
    return messages

def chat_con_documento(message, history, document_text):
    messages = [{"role": "system", "content": build_system_prompt(document_text)}]
    messages.extend(history_to_messages(history))
    messages.append({"role": "user", "content": message})
    stream = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )
    response = ""
    for chunk in stream:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta.content
        if delta:
            response += delta
            yield response

## Paso 4: interfaz Gradio

Construye la interfaz con `gr.ChatInterface`.

**Instrucciones:**

1. Extrae el texto del PDF con la función del paso 1.
2. Muestra cuántas páginas y caracteres se extrajeron.
3. Configura `gr.ChatInterface` con:
   - `fn=chat_con_documento`;
   - un título y una descripción;
   - un `gr.Textbox` oculto dentro de `additional_inputs` para enviar el documento;
   - al menos tres preguntas de ejemplo.

```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz:

```python
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)
```

In [57]:
import gradio as gr

gr.close_all()  # cierra cualquier servidor Gradio abierto en este kernel

pdf_path = "attention_is_all_you_need.pdf"
document_text = extract_text_from_pdf(pdf_path)

num_pages = len(PdfReader(pdf_path).pages)
num_chars = len(document_text)
print(f"El documento tiene {num_pages} páginas y cuenta con {num_chars} caracteres.")

preguntas_ejemplo = [
    "¿Cuál es el objetivo central del paper?",
    "¿Qué conclusiones puedes extraer del mismo?",
    "¿En qué idioma está escrito el documento?",
]

demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="Attention is all you need chat",
    description="Chatea con un asistente especializado que contestará específicamente en base a este documento",
    additional_inputs=[gr.Textbox(value=document_text, visible=False)],
    examples=[[q, document_text] for q in preguntas_ejemplo],
    cache_examples=False,
)

demo.launch(server_name="0.0.0.0", server_port=7880, show_error=True)

Closing server running on port: 7880
El documento tiene 15 páginas y cuenta con 39585 caracteres.
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9d1a14de1995683f35.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Paso 5: Prueba y reflexión

Una vez que la interfaz esté funcionando, prueba estas preguntas:

1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*

Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*

Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?
Solucion:
1. La arquitectura principal propuesta en el paper es el Transformer, una red de secuencias basada exclusivamente en mecanismos de atención que elimina por completo la recurrencia y las convoluciones. El Transformer consta de un codificador y un decodificador apilados (N = 6 capas idénticas cada uno), donde cada capa contiene una sub‑capas de atención multi‑cabeza seguida de una red feed‑forward punto‑a‑punto, todo ello con conexiones residuales y normalización de capa. Como se menciona en el resumen: “We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.”【0†lines 9‑12】. Esta arquitectura permite un alto grado de paralelización y logra un nuevo estado‑de‑arte en tareas de traducción automática.
 2. El mecanismo de atención es una función que mapea una consulta y un conjunto de pares clave-valor a una salida, donde la consulta, claves, valores y salida son todos vectores. La salida se calcula como una suma ponderada de los valores, donde el peso asignado a cada valor se calcula mediante una función de compatibilidad de la consulta con la correspondiente clave. (Sección 3.2)
 3. Según el documento, en la sección 3.1 Encoder and Decoder Stacks se especifica explícitamente: "Encoder: The encoder is composed of a stack of N = 6 identical layers." Esto corresponde al modelo base, como se confirma en la Tabla 3 donde la fila correspondiente al "base model" muestra N = 6.
6 capas
4. Los autores del paper "Attention Is All You Need" son:

    Ashish Vaswani∗ (Google Brain)
    Noam Shazeer∗ (Google Brain)
    Niki Parmar∗ (Google Research)
    Jakob Uszkoreit∗ (Google Research)
    Llion Jones∗ (Google Research)
    Aidan N. Gomez∗ † (University of Toronto)
    Łukasz Kaiser∗ (Google Brain)
    Illia Polosukhin∗ ‡ (illia.polosukhin@gmail.com)

    Nota: Los símbolos ∗ indican contribución igual, † indica trabajo realizado mientras estaba en Google Brain, y ‡ indica trabajo realizado mientras estaba en Google Research. (Como se menciona en el documento: "∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started the effort to evaluate this idea. Ashish, with Illia, designed and implemented the first Transformer models and has been crucially involved in every aspect of this work. Noam proposed scaled dot-product attention, multi-head attention and the parameter-free position representation and became the other person involved in nearly every detail. [...] †Work performed while at Google Brain. ‡Work performed while at Google Research.")
5. 28.4 BLEU
6. No encuentro esa información en el documento.

## Paso 6 (adicional): mejora el sistema

Si terminan antes, implementen una mejora:

**A) Medición del contexto**  
Reporta la cantidad de caracteres o estima los tokens del documento. Explica
por qué caracteres y tokens no son equivalentes.

**B) Subida dinámica de PDF**  
Agrega `gr.File` para cargar cualquier PDF y extrae el texto dentro del flujo de
la aplicación.

**C) Evidencia y páginas**  
Conserva separadores de página al extraer el PDF y exige que cada respuesta cite
la página que contiene la evidencia.

**D) Primer paso hacia RAG**  
Divide el documento en fragmentos y selecciona los más relevantes mediante una
búsqueda simple por palabras antes de llamar al modelo.

## Reflexión final

Discutan en grupo:

1. ¿Cuál es la limitación de enviar siempre el documento completo?
2. ¿Por qué este ejercicio todavía no implementa RAG?
3. ¿Cómo comprobarían que la respuesta proviene del documento y no del
   conocimiento previo del modelo?
4. ¿Qué riesgos aparecen al insertar directamente documentos de usuarios en el
   prompt?

Respuesta

1. La mayor limitacion es que al enviar el documento tantas veces se guarda vez tras vez en el historial, lo que causa que la memoria se llene mas rapido y reduce los tokens disponibles
2. Todavia no implementamos RAG debido a que primero deberiamos conocer las bases de este tipo de extraccion, ya que RAG es mucho mas compleja
3. Al ponerle los limitantes a la hora de responder, se le especifico que unicamente basara su respuesta en el documento, asi como tambien realizar citas textuales todo lo posible, por lo que seria mas facil verificar la parte especifica a la que se esta refiriendo
4. El programa no distingue bien en que momento inician sus instrucciones y en que momento inicia el documento que le insertamos, por lo que podria divagar en sus respuestas o ignorar algunas restricciones

## Recursos

- [pypdf](https://pypdf.readthedocs.io)
- [Gradio](https://www.gradio.app/docs)
- [NVIDIA Build](https://build.nvidia.com)
- [Nemotron 3 Super](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b)
- [Paper original](https://arxiv.org/abs/1706.03762)